In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

## PyTorch: Higher-level abstractions

* Structured layers: ``torch.nn.module``
* Standard optimizers ``torch.optim`` 
* Datasets and data loaders
* The functional API

## Motivation

Let us define a linear model manually

$$ f: \mathbb{R}^{10} \rightarrow \mathbb{R}^{4} := Wx + b $$

In [ ]:
nx = 10; ny = 4;
W = torch.randn(ny, nx, requires_grad=True); b = torch.randn(ny, requires_grad=True)
x = torch.randn(nx)

def lin(W, b, x):
    y = W @ x + b
    return y 

x.shape, lin(W, b, x).shape

Only work on a **single** instance of size $n_x$. For a batch of size $(B, n_x)$, we need something else:

In [ ]:
batch_size = 32; batch_x = torch.randn(batch_size, nx)

def batched_lin(W, b, batch_x):
    y =  batch_x @ W.T + b
    return y

batch_x.shape, batched_lin(W, b, batch_x).shape

It is much easier to use the built-in PyTorch Linear module

In [ ]:
lin = nn.Linear(in_features=nx, out_features=ny, bias=True) # bias=True -> we have the b term

In [ ]:
lin.weight.shape, lin.bias.shape # W.T and b

The module can be called with data, either on a single instance or on a batch

In [ ]:
x.shape, batch_x.shape

In [ ]:
lin(x).shape, lin(batch_x).shape

Advantages of the ``nn.Linear`` module

* Model and parameters within a single object
* Handles single instances and batches seamlessly
* Less code, more readable

The biggest advantages are for more complex models! 

## Model construction

We can build more complex models by subclassing ``nn.Module``. 

Let us define a MLP with 10 inputs, 4 outputs, 100 hidden units:

$$f: \mathbb{R}^{10} \rightarrow \mathbb{R}^{4} := W_2 \tanh (W_1 x + b_1) + b_2$$

$$
W_1 \in \mathbb{R}^{100 \times 10}, b_1 \in \mathbb{R}^{100}, W_2 \in \mathbb{R}^{4 \times 100}, b_2 \in \mathbb{R}^{10},
$$

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_size, out_size, hidden_size):
        super().__init__()
        self.lin1 = nn.Linear(in_size, hidden_size)
        self.activation = nn.Tanh()
        self.lin2 = nn.Linear(hidden_size, out_size)

    def forward(self, x):
        x = self.activation(self.lin1(x)) # hidden layer
        x = self.lin2(x) # output layer
        return x

In [ ]:
mlp = MLP(in_size=10, out_size=4, hidden_size=100); mlp

In [ ]:
x.shape, batch_x.shape

In [ ]:
mlp(x).shape, mlp(batch_x).shape

## Example

The same of the previous exercise: data from

$$ y = 2 \sin(3x) - 3 \cos(7x) $$

In [ ]:
nx = 1; ny = 1; a = -1; b = 1; n_samples = 100; sigma_e = 0.1
x_train = a + torch.rand((n_samples, 1))*(b - a)
y_train = 2*torch.sin(3*x_train)  - 3*torch.cos(7*x_train)
plt.figure(figsize=(8, 3))
plt.plot(x_train, y_train, "*");

In [ ]:
mlp = MLP(in_size=1, out_size=1, hidden_size=16); mlp

In [ ]:
mlp(x_train).shape

## Training with ``nn.Module``

In [ ]:
mlp = MLP(in_size=1, out_size=1, hidden_size=16)
LOSS = []
for _ in range(10_000):
    y_hat = mlp(x_train)
    loss = torch.mean((y_train - y_hat)**2)
    loss.backward()
    # manual gradient descent
    for p in mlp.parameters():
        p.data -= 0.01 * p.grad
        p.grad.zero_() # otherwise gradients get accumulated
    LOSS.append(loss.item())

In [ ]:
y_hat = mlp(x_train).detach() 
fig, ax = plt.subplots(1, 2, figsize=(13, 3))
ax[0].plot(LOSS, "k*")
ax[1].plot(x_train, y_train, "k*")
ax[1].plot(x_train, y_hat, "b*")
ax[1].plot(x_train, y_train - y_hat, "r*");

## Training with ``nn.Module`` and PyTorch optimizers

* Plain gradient descent is not the best algorithm. Adam, AdamW, L-BFGS are often more effective
* It is boring and error-prone to implement optimizers from scratch
* PyTorch provides standard optimizers in ``torch.optim``

In [ ]:
mlp = MLP(in_size=1, out_size=1, hidden_size=16)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3) # ready-made optimizers

LOSS = []
for _ in range(10_000):
    opt.zero_grad() # cleans gradients accumulated from previous iterations
    y_hat = mlp(x_train)
    loss = torch.mean((y_train - y_hat)**2)
    loss.backward() # gradient computation
    opt.step() # optimization step
    LOSS.append(loss.item())

In [ ]:
y_hat = mlp(x_train).detach() # can't plot tensors in a computational graph - need to be detached!
fig, ax = plt.subplots(1, 2, figsize=(15, 3))
ax[0].plot(LOSS, "k*")
ax[1].plot(x_train, y_train, "k*")
ax[1].plot(x_train, y_hat, "b*")
ax[1].plot(x_train, y_train - y_hat, "r*");

## Datasets and data loaders

In the previous examples, the dataset was very small and we trained on all instances each iteration

* It is more common to train with (mini)batches
* Sometimes the data extraction logic is complex
* The ``Dataset`` and ``DataLoader`` classes in ``torch.utils.data`` facilitate data handling

We use a larger dataset to make things look a little bit more realistic:

In [ ]:
x_train_long = a + torch.rand((100*n_samples, 1))*(b - a)
y_train_long = 2*torch.sin(3*x_train_long)  - 3*torch.cos(7*x_train_long)
x_train_long.shape, y_train_long.shape

## Dataset

The PyTorch dataset is meant to hide the complexity of retreiving training instances

In [ ]:
from torch.utils.data import Dataset, TensorDataset

class MyDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    # number of training instances
    def __len__(self):
        return len(self.x)

    # logit to access a single instance.
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx] # this could be a complex data retrieval operation instead!

In [ ]:
ds = MyDataset(x_train_long, y_train_long)
ds[0] # a simple input-output pair

In [ ]:
x_train_long[0], y_train_long[0]

In ``MyDataset``, the ``__getitem__`` logic is trivial . In general, could hide file read, database access, synthetic data generation, ...

In [ ]:
ds = TensorDataset(x_train_long, y_train_long) # this built-in DataSet is equivalent to MyDataset
ds[0]

### Data Loader

The ``DataLoader`` takes in a ``Dataset`` and provides an iterator over the batches

In [ ]:
from torch.utils.data import DataLoader

ds = TensorDataset(x_train_long, y_train_long)
dl = DataLoader(ds, batch_size=32, shuffle=True)

In [ ]:
batch_x, batch_y = next(iter(dl)) # the data loader is a Python iterable
batch_x.shape, batch_y.shape # a batch of training instances, ready for a training step!

In particular, we can iterate it with a for loop, where iterative optimization will run. We typically pass through the dataset a number of *epochs*.

In [ ]:
for epoch in range(10):
    for (batch_x, batch_y) in dl:
        pass # actual training step here!

## Training Loop

Finally, this canonical training code in PyTorch!

In [ ]:
ds = TensorDataset(x_train_long, y_train_long)
dl = DataLoader(ds, batch_size=128, shuffle=True)
mlp = MLP(in_size=1, out_size=1, hidden_size=16)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-2)

LOSS = []
for epoch in range(40): # 1 epoch is a "pass" through the whole training dataset
    for (batch_x, batch_y) in dl: # 1 iteration is an optimization step executed on a minibatch

        opt.zero_grad()
        batch_y_hat = mlp(batch_x)
        loss = torch.mean((batch_y - batch_y_hat)**2)
        
        loss.backward() # gradient computation
        opt.step() # optimization step
        LOSS.append(loss.item())

In [ ]:
with torch.no_grad():
    y_hat = mlp(x_train)
    
fig, ax = plt.subplots(1, 2, figsize=(15, 3))
ax[0].plot(LOSS, "k*")
ax[1].plot(x_train, y_train, "k*")
ax[1].plot(x_train, y_hat, "b*")
ax[1].plot(x_train, y_train - y_hat, "r*");

# The Functional API


PyTorch "hide" parameters within modules. Often convenient, but limiting/awkward for some advanced use cases

In [ ]:
mlp

In [ ]:
batch_x = torch.randn(batch_size, 1); batch_y = torch.randn(batch_size, 1); 
mlp(batch_x).shape

We can optionally call modules as functions. First, extract the parameters to a dictionary of tensors:

In [ ]:
params = dict(mlp.named_parameters()) # dictionary parmeter_name : parameter_value

Then, call them with ``torch.func.functional_call``

In [ ]:
batch_y_pred = torch.func.functional_call(mlp, params, batch_x); batch_y_pred.shape

# The Functional API

We may then write the loss as a differentiable function...

In [ ]:
def loss_fn(params, x, y):
    y_pred = torch.func.functional_call(mlp, params, x);
    loss = torch.mean((y - y_pred)**2)
    return loss
    
loss_fn(params, batch_x, batch_y)

and the loss gradient as a differentiable function...

In [ ]:
grad_fn = torch.func.grad(loss_fn, argnums=0) # gradient of loss_fn wrt first argument
grads = grad_fn(params, batch_x, batch_y) # dictionary parameter_name: gradient_value
grads

and the parameter update as a differentiable function.

In [ ]:
def gd_update_fn(p, x, y, alpha):
    grad_fn = torch.func.grad(loss_fn, argnums=0)
    g = grad_fn(p, x, y)
    p_upd = {k: p[k] - alpha * g[k] for k in p.keys()} # gradient descent step
    return p_upd

In [ ]:
alpha = torch.tensor(0.1)
params_updated = gd_update_fn(params, batch_x, batch_y, alpha) # parameters updated by gradient descent!

The function ``gd_update_fn`` is a gradient descent step, and it is still differentiable with respect to all its arguments!

This allows one to optimize (parameterized) algorithms, e.g.
* learn the best initialization for gradient descent (MAML)
* learn the best learning rate by gradient descent

Learning algorithms automatically from data is known as **meta learning**, aka learning to learn.

## The Functional API

Another super useful tool in the functional API is ``vmap``, which allows you to vectorize ("batchify") functions.

In [ ]:
n_hidden = 16; nx=3; ny=2; batch_size=32
params = {
    "W1": torch.randn(n_hidden, nx, requires_grad=True),
    "b1": torch.randn(n_hidden, requires_grad=True),
    "W2": torch.randn(ny, n_hidden, requires_grad=True),
    "b2": torch.randn(ny, requires_grad=True)
}

def net(params, x):
    # shape of x: (nx,)
    z = torch.tanh(params["W1"] @ x + params["b1"])
    y = params["W2"] @ z + params["b2"]
    return y

In [ ]:
batched_net = torch.func.vmap(net, in_dims=(None, 0)) # vectorize with respect to the second argument (x) shape of x: (batch_size, nx)

In [ ]:
batched_net(params, torch.randn((batch_size, nx))).shape

This allows one to define functions on a single example, without the need of tricky linear algebra/broadcasting tricks

Without vmap

In [ ]:
# We can "vectorize" the nn function with linear algebra and broadcasting tricks
def batched_net(params, x):
    z = torch.sigmoid(x @ params["W1"].T + params["b1"])
    y = z @ params["W2"].T + params["b2"]
    return y

need linear algebra tricks to do the vectorization

With vmap:

In [ ]:
batched_net = torch.func.vmap(net, in_dims=(None, 0)) # vectorize with respect to the second argument (x) shape of x: (batch_size, nx)

you explicitly ask to vectorize the simpler function.

Functional programming with powerful transformations like ``vmap`` is the default approach in other deep learning libraries (eg, JAX).